# NPML Model Inventory and Gap Map

This notebook maps the five NPML talk experiments from the PDF to the code currently present in `src/`
and to the minimum additional work needed to run the architecture-heavy experiments honestly.

In [1]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from npml_support import *

dirs = ensure_npml_dirs()

In [2]:
plan_df = pdf_experiment_plan()
inventory_df = audit_src_models()
needs_df = experiment_need_table()

display(plan_df)
display(inventory_df)
display(needs_df)

save_dataframe(plan_df, "00_pdf_experiment_plan.csv")
save_dataframe(inventory_df, "00_src_model_inventory.csv")
save_dataframe(needs_df, "00_needed_work.csv")

,experiment,title,design,metrics,goal
0,A,Metric Ablation,"PCA vs EMPCA under white, pink, brownian, MMC PSD",weighted residual; reconstruction MSE; principal angle; amplitude resolution,Synthetic traces with colored Gaussian noise
1,B,Coverage Ablation,Full latent coverage vs timing/position/shape restricted training,amplitude RMSE; timing RMSE; position RMSE; weighted residual,"Train restricted, test always full"
2,C,NFPA vs EMPCA,Separable vs non-separable multichannel signals,weighted residual; reconstruction error; subspace angle,Tests channel-time factorization
3,D,Architecture Bias,Linear AE vs CNN AE vs Transformer AE,weighted residual; reconstruction RMSE; unseen position/shape generalization,Identical Mahalanobis objective
4,E,Prewhitened Transformer,raw/prewhitened x MSE/Mahalanobis,weighted residual; amplitude/timing resolution; generalization,Attention in detector geometry


,experiment,model_family,path,status,runnable_now,notes
0,A,PCA baseline,implementation.notebook_support exact_isotropic_subspace,ready,True,Exact isotropic SVD baseline exists in implementation helpers.
1,A/B,EMPCA,src/EMPCA/empca_TCY_optimized.py,ready,True,Weighted EMPCA solver is importable and already wired into implementation.notebook_support.
2,C,NFPA demo,src/NFPA/nfpa_demo.py,partial,True,"Pure numpy demo is runnable as a script, but it executes on import and hardcodes a plot write side effect."
3,D,Linear AE,implementation.notebook_support exact_weighted_subspace,partial,True,"Exact weighted SVD gives the linear-AE optimum, but there is no standalone training wrapper under src/."
4,D,CNN backbone,src/CNN/resnet_1d.py,blocked,False,"Torch installed? False. Local norm dependency present? False. File defines a ResNet backbone, not a reconstruction autoencoder or trainer."
5,D/E,model_original,src/transformer/model_original.py,blocked,False,"Requires torch and reconstruction.training.muon; torch installed=False, muon module present=True. These files also define transformer backbones only, not en..."
6,D/E,model_pairwise,src/transformer/model_pairwise.py,blocked,False,"Requires torch and reconstruction.training.muon; torch installed=False, muon module present=True. These files also define transformer backbones only, not en..."
7,D/E,model_pairwise_channel_masking,src/transformer/model_pairwise_channel_masking.py,blocked,False,"Requires torch and reconstruction.training.muon; torch installed=False, muon module present=True. These files also define transformer backbones only, not en..."
8,D/E,model_triangular_pairwise,src/transformer/model_triangular_pairwise.py,blocked,False,"Requires torch and reconstruction.training.muon; torch installed=False, muon module present=True. These files also define transformer backbones only, not en..."
9,A,Measured MMC PSD,data/Noise_PSD/noise_psd_from_MMC.npy,ready,True,The measured MMC PSD is available in the new data/Noise_PSD layout.


,experiment,need
0,A,Add measured MMC PSD file if you want the fourth noise condition exactly as written in the PDF.
1,B,Implement a native position latent in the simulator if you want physically grounded position RMSE rather than a proxy distortion.
2,C,Refactor src/NFPA/nfpa_demo.py into callable functions if you want it reusable outside a demo script.
3,D,"Add a reconstruction training stack: linear AE wrapper, CNN AE wrapper, shared Mahalanobis loss, and evaluation harness."
4,D,Install torch and provide the missing local dependency used by src/CNN/resnet_1d.py.
5,E,"Install torch, provide reconstruction.training.muon, and add transformer reconstruction heads plus a whitening pipeline."
6,E,"Add four controlled training configs: raw+mse, raw+mahalanobis, whitened+mse, whitened+mahalanobis."


PosixPath('/Users/wongdowling/Documents/noise-weighted-subspace-reconstruction/paper2/npml/tables/00_needed_work.csv')

In [3]:
fig = plot_readiness(inventory_df, "NPML experiment readiness from current src/ inventory")
save_figure(fig, "00_readiness_overview.png")
plt.show()
plt.close(fig)

/var/folders/y6/2zvmjscx2s18w96j7qx32nnr0000gn/T/ipykernel_23017/1742797722.py:3: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
